In [2]:
# prepare_from_folders.py
import os, json, re, shutil, random
from pathlib import Path
from PIL import Image
from tqdm import tqdm

# ----- 사용자 경로 -----
IMG_DIR = Path(r"/Users/qwergs/Desktop/데이터크리에이터 캠프/data/53.대기오염 배출원 공간 분포 데이터/3.개방데이터/1.데이터/Training/01.원천데이터/TS_KS")
LBL_DIR = Path(r"/Users/qwergs/Desktop/데이터크리에이터 캠프/data/53.대기오염 배출원 공간 분포 데이터/3.개방데이터/1.데이터/Training/02.라벨링데이터/TL_KS_BBOX")

# ----- 출력 폴더 -----
# WORKDIR   = Path.cwd() / "chimney_yolo"
BASE_DIR = Path(r"/Users/qwergs/Desktop/chimney_project")  # 원하는 폴더로 변경 가능
WORKDIR  = BASE_DIR / "chimney_yolo"
DS_DIR    = WORKDIR / "dataset"           # train/val/images,labels
YAML_FILE = WORKDIR / "data.yaml"
random.seed(42)

IMG_EXTS = {".jpg",".jpeg",".png",".tif",".tiff",".bmp"}

def safe_mkdir(p: Path): p.mkdir(parents=True, exist_ok=True)

def find_images(root: Path):
    return [p for p in root.rglob("*") if p.suffix.lower() in IMG_EXTS]

def to_yolo_xywhn(x, y, w, h, W, H):
    cx = (x + w/2) / W
    cy = (y + h/2) / H
    wn = w / W
    hn = h / H
    return (max(0,min(1,cx)), max(0,min(1,cy)),
            max(0,min(1,wn)), max(0,min(1,hn)))

# 라벨 JSON 파서: (1) {"shape_attributes":{...}}
#                 (2) VIA 형식 {"regions":[{"shape_attributes":{...}}, ...]}
#                 (3) {"bbox":{"x":..,"y":..,"width":..,"height":..}}
def extract_rects(d: dict):
    rects = []

    def pick_rect(sd):
        if not isinstance(sd, dict): return
        name = (sd.get("name") or sd.get("type") or "").lower()
        if name and ("rect" not in name and "box" not in name): 
            return
        x = sd.get("x"); y = sd.get("y")
        w = sd.get("width") or sd.get("w")
        h = sd.get("height") or sd.get("h")
        if None not in (x,y,w,h):
            rects.append((float(x),float(y),float(w),float(h)))

    if "shape_attributes" in d:           # case (1)
        pick_rect(d["shape_attributes"])

    if "regions" in d and isinstance(d["regions"], (list,dict)):  # case (2)
        regs = d["regions"].values() if isinstance(d["regions"], dict) else d["regions"]
        for r in regs:
            pick_rect(r.get("shape_attributes", {}))

    if "bbox" in d and isinstance(d["bbox"], dict):               # case (3)
        pick_rect(d["bbox"])

    # 혹시 문자열로 저장된 경우를 대비
    for k in ["shape_attributes","region_shape_attributes"]:
        v = d.get(k)
        if isinstance(v, str) and v.strip().startswith("{"):
            try:
                pick_rect(json.loads(v))
            except:
                pass

    return rects

def write_yaml(path: Path, ds_root: Path):
    path.write_text(
f"""path: {ds_root}
train: train/images
val: val/images

names:
  0: chimney
"""
    )

def main():
    # iCloud(구름 아이콘)로 미다운로드된 파일이 있으면 학습이 실패합니다.
    # Finder에서 폴더 우클릭 → "즉시 다운로드"로 모두 로컬화하세요.

    if WORKDIR.exists():
        shutil.rmtree(WORKDIR)
    safe_mkdir(WORKDIR)

    # 1) 이미지 리스트 수집
    images = find_images(IMG_DIR)
    if not images:
        raise RuntimeError("이미지 파일을 찾지 못했습니다.")
    print(f"[IMAGES] {len(images)} found")

    # 2) 파일명 기준 JSON 매칭 → YOLO txt 생성용 임시 맵 구성
    json_map = {p.stem: p for p in LBL_DIR.rglob("*.json")}

    # 3) train/val 디렉터리 만들기
    tr_img = DS_DIR / "train" / "images"; tr_lbl = DS_DIR / "train" / "labels"
    va_img = DS_DIR / "val"   / "images"; va_lbl = DS_DIR / "val"   / "labels"
    for p in [tr_img,tr_lbl,va_img,va_lbl]: safe_mkdir(p)

    # 4) split
    random.shuffle(images)
    cut = int(len(images)*0.9)
    train_set, val_set = images[:cut], images[cut:]

    def process_split(split_imgs, dst_img, dst_lbl):
        for ip in tqdm(split_imgs, desc=f"build {dst_img.parts[-2]}"):
            stem = ip.stem
            jp = json_map.get(stem) or json_map.get(stem.replace(" ", "_"))
            # 이미지 크기
            with Image.open(ip) as im:
                W,H = im.size

            # YOLO 라벨 파일 경로
            ypath = dst_lbl / f"{stem}.txt"
            ys = []
            if jp and jp.exists():
                try:
                    d = json.loads(jp.read_text())
                    rects = extract_rects(d)
                    for (x,y,w,h) in rects:
                        cx,cy,wn,hn = to_yolo_xywhn(x,y,w,h,W,H)
                        ys.append(f"0 {cx:.6f} {cy:.6f} {wn:.6f} {hn:.6f}")
                except Exception as e:
                    print(f"[WARN] JSON parse failed: {jp.name} ({e})")

            # 라벨 있으면 저장(없으면 배경 이미지로 사용)
            if ys:
                ypath.write_text("\n".join(ys))

            # 이미지 복사
            shutil.copy2(ip, dst_img / ip.name)

    process_split(train_set, tr_img, tr_lbl)
    process_split(val_set,   va_img, va_lbl)

    # 5) data.yaml
    write_yaml(YAML_FILE, DS_DIR)

    print("\n[READY]")
    print(" data.yaml :", YAML_FILE)
    print(" train imgs:", len(list(tr_img.glob('*'))))
    print(" val   imgs:", len(list(va_img.glob('*'))))
    print("\n다음 명령으로 학습을 시작하세요:")
    print(f"  yolo detect train data='{YAML_FILE}' model=yolov8n.pt imgsz=1024 epochs=50 batch=16 workers=4")

if __name__ == "__main__":
    main()

[IMAGES] 8052 found


build val: 100%|██████████| 806/806 [00:00<00:00, 939.03it/s]


[READY]
 data.yaml : /Users/qwergs/Desktop/chimney_project/chimney_yolo/data.yaml
 train imgs: 7246
 val   imgs: 806

다음 명령으로 학습을 시작하세요:
  yolo detect train data='/Users/qwergs/Desktop/chimney_project/chimney_yolo/data.yaml' model=yolov8n.pt imgsz=1024 epochs=50 batch=16 workers=4
